# Learning by Trial and Error: Reinforcement Learning for Managers
### A companion to *Supply Chain AI in Python* · scmaiteaching.org · Executive track

*A companion extra, written for the site. It is not part of the book.*

**License:** the code in this notebook is under the MIT License (full text in the last cell). It covers the code only, not the explanatory text. © 2026 Charles C. Luo.

**Who this is for.** The book's Executive track (Appendix I, Table I.1): "MBA students, managers with limited coding experience", who "Run provided notebooks as-is; focus on interpreting outputs, dashboards, and strategic implications". You will not write any code. You will run cells, read charts and decide what they mean for a business.

**The situation.** Priya runs replenishment for a regional grocery chain *(a constructed example)*. A vendor offers her "an AI that learns your ordering policy by itself". Before she can judge the offer she needs three answers: what "learning by itself" means, what it costs, and when it beats the rule her planners already use. This notebook answers those questions with a dog, a maze and a treat, and then brings the answers back to the supply chain.

**How each part works** (about 8–10 minutes a part, under an hour in all):

| Step | What you do |
|---|---|
| **Read** | A short explanation, in plain language |
| **Run** | Press *Shift + Enter* on the code cell. No coding needed |
| **Notice** | What the chart shows, with the settings as shipped |
| **Discuss** | One or two questions for your class or team |
| **Takeaway** | The point to keep, as a manager |

The parts build one story: the same dog meets harder mazes, and each part adds one idea. Short on time? Each part opens with **the two-minute version**.

**First time in Colab?**

1. **Sign in.** If you aren't signed in to Google, choose *Sign in* (top right). If your school account can't open Colab, use a personal one.
2. **Run everything.** Choose *Runtime → Run all*. Colab warns that the notebook was not authored by Google; this one installs nothing and asks for no access to your files, so choose *Run anyway*.
3. **Wait for each cell to finish.** A green check mark appears beside a cell once it has run. The whole notebook takes under a minute.
4. **Run one cell.** Click inside it and press *Shift+Enter*. Go from top to bottom: a later cell uses what an earlier one made.
5. **If a cell fails**, for example with a `NameError` after a break (Colab clears the session when it disconnects) or after running cells out of order, choose *Runtime → Restart session and run all*.
6. **Keep your changes.** Edits to the shared notebook are not saved for you. Choose *File → Save a copy in Drive*. The copy opens in a new tab; it is yours, so keep working there, starting with *Runtime → Run all*.

Then change the settings marked `# try changing this` and run again. The checks run only on the settings as shipped; once you change one, they are skipped, because the shipped conclusions may no longer hold — that is the point of experimenting.

---
## The five words you need

| Word | In plain language | In the maze | In a replenishment system |
|---|---|---|---|
| **Agent** | The one who decides | The dog | The ordering system |
| **State** | What it knows when it decides | Where it is standing | Stock on hand, orders on the way, recent demand, the forecast and more (Sect. 3.4.2 lists them) |
| **Action** | What it decides | Which way to go | How much to order |
| **Reward** | The score it gets afterward | A treat at the end; a small cost for every step | Minus this period's holding, shortage and ordering cost |
| **Policy** | Its habit: what it does in each situation | Which way it goes from each spot | The reorder rule |

One complete try, from start to finish, is a **trial** (in the maze, one walk to the treat; in a business, one simulated season). The book also calls it an *episode*.

*Reinforcement learning (RL) is a computing method. The dog is an analogy that makes the method easy to picture; it is not a claim about how dogs learn.*

In [ ]:
# Run this cell first. You do not need to read it.
# Helper code for "Learning by Trial and Error: Reinforcement Learning for Managers".
# You do not need to read this cell to follow the notebook: run it and move on.
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

MOVES = {"up": (-1, 0), "down": (1, 0), "left": (0, -1), "right": (0, 1)}
NAMES = list(MOVES)
ARROW = {"up": (0, -0.35), "down": (0, 0.35), "left": (-0.35, 0), "right": (0.35, 0)}


class Maze:
    """A grid maze. '#' wall, '.' floor, 'S' start, 'T' treat, 'K' small treat, 'M' mud, 'A'/'B' possible treat spots."""

    def __init__(self, rows, step_cost=-1.0, treat=10.0, small=3.0, mud_cost=0.0, max_steps=200):
        self.grid = [list(r) for r in rows]
        self.h, self.w = len(rows), len(rows[0])
        self.step_cost, self.treat, self.small, self.mud_cost, self.max_steps = step_cost, treat, small, mud_cost, max_steps
        find = lambda ch: [(r, c) for r in range(self.h) for c in range(self.w) if self.grid[r][c] == ch]
        self.start = find("S")[0]
        self.treats = {p: treat for p in find("T")}
        self.treats.update({p: small for p in find("K")})
        self.mud = set(find("M"))
        self.spots = find("A") + find("B")            # for the 'moving treat' maze

    def open(self, p):
        r, c = p
        return 0 <= r < self.h and 0 <= c < self.w and self.grid[r][c] != "#"

    def step(self, p, a, goal=None):
        dr, dc = MOVES[a]
        q = (p[0] + dr, p[1] + dc)
        if not self.open(q):
            q = p                                      # bumping into a wall: stay put
        reward = self.step_cost - (self.mud_cost if q in self.mud else 0.0)
        goals = {goal: self.treat} if goal is not None else self.treats
        done = q in goals
        if done:
            reward += goals[q]
        return q, reward, done


def shortest_steps(maze, start, targets):
    """Fewest moves from start to any target (breadth-first search): the yardstick."""
    seen, queue = {start: 0}, deque([start])
    while queue:
        p = queue.popleft()
        if p in targets:
            return seen[p]
        for a in NAMES:
            q, _, _ = maze.step(p, a)
            if q not in seen:
                seen[q] = seen[p] + 1
                queue.append(q)
    return None


def random_walk(maze, runs=200, seed=0):
    """Steps a dog needs when it wanders at random until it finds a treat."""
    rng = np.random.default_rng(seed)
    steps = []
    for _ in range(runs):
        p, n, done = maze.start, 0, False
        while not done and n < 10_000:
            p, _, done = maze.step(p, NAMES[rng.integers(4)])
            n += 1
        steps.append(n)
    return np.array(steps)


def q_learning(maze, trials=300, alpha=0.5, gamma=0.95, explore=0.1, seed=0, sees_goal=False):
    """Tabular Q-learning. Returns the learned scores and the steps taken in each trial.
    sees_goal=True lets the learner know which spot the treat is in this trial (the 'moving treat' maze)."""
    rng = np.random.default_rng(seed)
    Q = {}
    steps_per_trial = []
    for _ in range(trials):
        goal = maze.spots[rng.integers(len(maze.spots))] if maze.spots else None
        p, n, done = maze.start, 0, False
        while not done and n < maze.max_steps:
            s = (p, goal) if sees_goal else p
            q_s = Q.setdefault(s, np.zeros(4))
            if rng.random() < explore:
                a = rng.integers(4)
            else:
                best = np.flatnonzero(q_s == q_s.max())
                a = rng.choice(best)                   # ties broken at random
            p2, r, done = maze.step(p, NAMES[a], goal)
            s2 = (p2, goal) if sees_goal else p2
            target = r if done else r + gamma * Q.setdefault(s2, np.zeros(4)).max()
            q_s[a] += alpha * (target - q_s[a])
            p, n = p2, n + 1
        steps_per_trial.append(n)
    return Q, np.array(steps_per_trial)


def follow(maze, Q, goal=None, sees_goal=False, limit=None):
    """The route the learner takes once it stops exploring: always its best-scored move."""
    limit = limit or maze.max_steps
    p, path, done = maze.start, [maze.start], False
    while not done and len(path) <= limit:
        s = (p, goal) if sees_goal else p
        q_s = Q.get(s, np.zeros(4))
        p, _, done = maze.step(p, NAMES[int(np.argmax(q_s))], goal)
        path.append(p)
    return path, done


def nose_rule(maze, goal, limit=None):
    """The simple rule, the incumbent: 'follow your nose'. Step to the neighboring open square that is fewest blocks
    from the treat, counted along the grid as if there were no walls; never step straight back unless there is no
    other way. It knows where the treat is."""
    limit = limit or maze.max_steps
    p, prev, path = maze.start, None, [maze.start]
    while p != goal and len(path) <= limit:
        options = [q for q in (maze.step(p, a)[0] for a in NAMES) if q != p]
        forward = [q for q in options if q != prev] or options
        q = min(forward, key=lambda q: (abs(q[0] - goal[0]) + abs(q[1] - goal[1]), q))
        prev, p = p, q
        path.append(p)
    return path, p == goal


def explore_then_commit(maze, explore_trials, commit_trials=100, seed=0):
    """Wander at random for a number of trials (exploring), then keep learning without ever exploring again."""
    rng_seed = seed
    Q, s1 = q_learning(maze, trials=explore_trials, explore=1.0, seed=rng_seed) if explore_trials else ({}, np.array([], int))
    Q2, s2 = q_learning_continue(maze, Q, trials=commit_trials, seed=rng_seed + 10_000)
    return Q2, s1, s2


def q_learning_continue(maze, Q, trials, alpha=0.5, gamma=0.95, seed=0):
    """Continue Q-learning from existing scores, never exploring (always the best-scored move, ties at random)."""
    rng = np.random.default_rng(seed)
    Q = {k: v.copy() for k, v in Q.items()}
    steps = []
    for _ in range(trials):
        p, n, done = maze.start, 0, False
        while not done and n < maze.max_steps:
            q_s = Q.setdefault(p, np.zeros(4))
            a = rng.choice(np.flatnonzero(q_s == q_s.max()))
            p2, r, done = maze.step(p, NAMES[a])
            target = r if done else r + gamma * Q.setdefault(p2, np.zeros(4)).max()
            q_s[a] += alpha * (target - q_s[a])
            p, n = p2, n + 1
        steps.append(n)
    return Q, np.array(steps)


def draw(maze, paths=(), title="", ax=None, goal=None):
    """Draw the maze, and any routes on it."""
    ax = ax or plt.subplots(figsize=(4.2, 4.2))[1]
    img = np.ones((maze.h, maze.w, 3))
    for r in range(maze.h):
        for c in range(maze.w):
            if maze.grid[r][c] == "#":
                img[r, c] = (0.25, 0.25, 0.30)
            elif (r, c) in maze.mud:
                img[r, c] = (0.72, 0.58, 0.40)
    ax.imshow(img)
    labels = {maze.start: "Start"}
    labels.update({p: ("Treat" if v == maze.treat else "Snack") for p, v in maze.treats.items()})
    for i, p in enumerate(maze.spots):
        labels[p] = "A" if i == 0 else "B"
    if goal is not None:
        labels[goal] = "Treat"
    if maze.mud:
        mid = sorted(maze.mud)[len(maze.mud) // 2]
        labels.setdefault(mid, "mud")
    for (r, c), text in labels.items():
        ax.text(c, r, text, ha="center", va="center", fontsize=8, fontweight="bold",
                color="#1b5e20" if text in ("Treat", "Snack") else ("#4e342e" if text == "mud" else "#0d47a1"))
    styles = [("#d32f2f", "-"), ("#1565c0", "--"), ("#2e7d32", ":")]
    for (label, path), (col, ls) in zip(paths, styles):
        ys, xs = zip(*path)
        ax.plot(xs, ys, color=col, linestyle=ls, linewidth=2.2, alpha=0.85, label=f"{label} ({len(path) - 1} steps)")
    if paths:
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), fontsize=8, frameon=False)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
    return ax


---
## 1. What is reinforcement learning, anyway?

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

A dog in a strange maze, with no map, finds the treat only by trying. In this maze, wandering at random takes about thirty times the shortest route. Reinforcement learning is learning a *sequence* of decisions from a score that may come only at the end, with no right answers to copy.
</details>

**Read.** Put a dog in a maze it has never seen, with a treat somewhere inside. Nobody shows it the map. It only finds out it did well when it reaches the treat. If it is put back at the start again and again, it can get faster by doing more of what worked.

That is the idea of reinforcement learning: **learning a sequence of decisions by trial and error, from a score that may come only at the end.** Compare it with the learning most forecasting uses, which learns from past examples where the right answer is known (Appendix A.2, *Learning from Examples*). Here there are no right answers to copy, only a score after acting, and today's move changes where you stand tomorrow.

**Run.** First, the dog without any learning: it wanders at random until it stumbles on the treat.

In [ ]:
MAZE_1 = ["#########",
          "#S..#...#",
          "#.#.#.#.#",
          "#.#...#.#",
          "#.#####.#",
          "#...#..T#",
          "###.#.###",
          "#.......#",
          "#########"]
maze1 = Maze(MAZE_1)
best = shortest_steps(maze1, maze1.start, set(maze1.treats))
wander = random_walk(maze1, runs=500, seed=0)

draw(maze1, title="Maze 1: from Start to the Treat"); plt.show()
print(f"Shortest route to the treat:              {best} steps")
print(f"A dog wandering at random (500 tries):     {wander.mean():.0f} steps on average, median {np.median(wander):.0f}")
print(f"That is about {wander.mean() / best:.0f} times the shortest route.")
assert wander.mean() > 20 * best          # 'more than twenty times the shortest route'

**Notice.**
- In this maze, wandering at random takes hundreds of steps, more than twenty times the shortest route.
- Without learning, finding the treat is slow and costly.

**Discuss.** In your business, what is the equivalent of the dog's wandering: decisions made without learning from what happened last time?

> **Takeaway.** RL is about *sequences* of decisions, where each choice changes the next situation, and where the only feedback is a score, often late.

---
## 2. Learning from experience

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

The dog keeps a score for every move at every spot and nudges it after each try. In this small maze, within a few dozen trials it walks a shortest route it was never shown; real problems need far more practice. Its first trials were long and costly, though; in a business they would be stockouts and overstock. That is why an RL system practices in a simulator first.
</details>

**Read.** Now let the dog learn. It keeps a score for every move at every spot ("how good is turning left here?"). After each move it nudges that score toward what happened next: the treat, or the step's small cost plus how good the next spot looks. Over many trials the moves on good routes earn high scores, and the habit becomes simple: *at each spot, take the best-scored move.* Most of the time it follows its habit; one move in ten it tries something at random, in case there is a better way. This method is **Q-learning**, the one the book's Chap. 5 notebook uses on a larger grid. (The students' intro, *Reinforcement Learning: A First Look*, shows the formula; you do not need it here.)

**Run.** Twenty dogs, each learning for 300 trials.

In [ ]:
DOGS, TRIALS = 20, 300                    # try changing this (TRIALS)
curves, routes = [], []
for dog in range(DOGS):
    Q, steps = q_learning(maze1, trials=TRIALS, seed=dog)
    curves.append(steps)
    routes.append(follow(maze1, Q))
curves = np.array(curves)
average = curves.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(np.arange(1, TRIALS + 1), average, color="#d32f2f")
axes[0].axhline(best, color="grey", linestyle="--", label=f"shortest route ({best} steps)")
axes[0].set_ylim(0, 200); axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Steps to the treat (average of the dogs)")
axes[0].set_title("The dogs get faster"); axes[0].legend()
draw(maze1, paths=[("route dog 1 learned", routes[0][0])], title="What dog 1 learned", ax=axes[1])
plt.tight_layout(); plt.show()

first, last = curves[:, :10].mean(), curves[:, -50:].mean()
lengths = sorted({len(path) - 1 for path, ok in routes if ok})
print(f"Average steps in the first 10 trials: {first:.0f}   in the last 50 trials: {last:.1f}")
print(f"Route each dog follows once it stops trying random moves: {lengths} steps (shortest possible: {best})")

SHIPPED = (DOGS, TRIALS) == (20, 300)
if SHIPPED:
    smooth = np.convolve(average, np.ones(10) / 10, mode="valid")
    near = int(np.argmax(smooth <= 1.25 * best)) + 10      # first trial whose 10-trial average is within 25%
    assert all(ok for _, ok in routes) and lengths == [best], "every dog should learn a shortest route"
    assert first > 4 * last, "the first trials should be far longer than the last"
    assert smooth.min() <= 1.25 * best and near <= 50, "close to the shortest route within a few dozen trials"
    assert last > best, "the last trials still average a little above the shortest route"
else:
    print("Checks skipped: you changed the settings.")

**Notice.**
- The first trials are long and costly; within a few dozen trials the dogs are close to the shortest route.
- Every dog learned a shortest route, without ever being shown the map.
- The last trials still average a little above the shortest route, because the dogs keep trying a random move now and then.
- This maze is tiny. Real problems need far more practice: the book's delivery-grid agent trained for 3,000 episodes (Chap. 5), and the inventory training loop in Sect. 3.4.4 ends "Repeat for millions of episodes".

**Discuss.** Who pays for the early, expensive trials in a real supply chain: stockouts, piles of overstock, unhappy customers? Could you afford them on your live system?

> **Takeaway.** RL buys a good policy with trials, and the early ones are bad. That is why the book trains its agents in a simulator: "The RL agent learns by interacting with a simulated inventory system" (Sect. 3.4.4). A key question to ask of any RL proposal is *where will it practice?*

---
## 3. Explore or exploit: settle for the snack, or look for the treat?

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

A snack is close; a better treat is further on. A dog that never explores eats the snack forever. One that wanders on purpose for a while finds the treat: here 50 trials of exploring were enough, and 200 bought nothing more. How much to explore, and where, is a management decision.
</details>

**Read.** A new maze. A small snack sits two steps from the start; a much better treat sits six steps away, down a side corridor. A dog that takes the first thing that works will find the snack and eat it every time. A dog that spends some trials **exploring** (wandering on purpose before it commits) may discover the treat. **Exploiting** means using what you already know. In this maze a step costs the dog nothing, so exploring costs only time: the steps spent wandering.

Every manager knows this trade-off: keep the supplier that works, or pilot a new one; keep the current policy, or test an alternative.

**Run.** Forty dogs for each setting. Each wanders at random for a number of trials, then commits: from then on it keeps learning but never explores again.

In [ ]:
MAZE_2 = ["########",
          "#K.S...#",
          "######.#",
          "#####T.#",
          "########"]
maze2 = Maze(MAZE_2, step_cost=0.0, treat=10.0, small=3.0)      # in this maze, steps are free
EXPLORE = [0, 20, 50, 200]                # try changing this: trials spent exploring before committing
share, snack, wandered = [], [], []
for n in EXPLORE:
    ends, steps_spent = [], []
    for dog in range(40):
        Q, exploring, committed = explore_then_commit(maze2, n, seed=dog)
        path, ok = follow(maze2, Q)
        ends.append(maze2.treats.get(path[-1]) if ok else None)
        steps_spent.append(exploring.sum())
    share.append(np.mean([e == maze2.treat for e in ends]))
    snack.append(np.mean([e == maze2.small for e in ends]))
    wandered.append(np.mean(steps_spent))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
draw(maze2, title="Maze 2: a snack nearby, a treat further on", ax=axes[0])
axes[1].bar([str(n) for n in EXPLORE], share, color="#2e7d32")
axes[1].set_ylim(0, 1.05); axes[1].set_xlabel("Trials spent exploring before committing")
axes[1].set_ylabel("Share of dogs that end up at the treat"); axes[1].set_title("A little exploring goes a long way")
plt.tight_layout(); plt.show()
for n, s, k, w in zip(EXPLORE, share, snack, wandered):
    print(f"explore {n:>3} trials: {s:4.0%} end up at the treat, {k:4.0%} at the snack; steps spent exploring: {w:6.0f} on average")

SHIPPED = EXPLORE == [0, 20, 50, 200]
if SHIPPED:
    assert share[0] < 0.2 and snack[0] >= 0.9, "with no exploring, almost every dog settles for the snack"
    assert share[1] > 0.5, "twenty trials of exploring get most dogs to the treat"
    assert share[2] == 1.0 and share[3] == 1.0, "fifty get every dog there, and 200 add nothing"
    assert wandered[0] < wandered[1] < wandered[2] < wandered[3], "the steps spent exploring grow"
else:
    print("Checks skipped: you changed the settings.")

**Notice.**
- With no exploring, almost every dog settles for the snack. It works, so it never looks further.
- Twenty trials of exploring get most dogs to the treat; fifty get every dog there. (Here steps are free; where every step costs something, as in the other mazes, twenty trials would get fewer dogs there.)
- Exploring is not free: the steps spent wandering grow with every exploring trial, and in this maze exploring for 200 trials buys nothing that 50 did not.

**Discuss.** Where does your organization "settle for the snack"? How would you decide how much to spend on exploring, and where (in a simulator, or on a small part of the live business)?

> **Takeaway.** A learning system must be allowed to try things that look worse in the short run. How much to explore, and where, is a management decision, not a technical detail.

---
## 4. What you reward is what you get

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

The shortest route crosses mud. If mud costs nothing, the dog takes the shortcut; once mud costs enough, it goes the clean way around. A learning system optimizes exactly what it is paid for, which is why the book leaves the price of side effects to the business.
</details>

**Read.** The shortest route to the treat now crosses a patch of mud. The dog does not mind mud; its owner does (bath time). Unless the reward charges for mud, the dog will take the muddy shortcut every time, because nothing in its score says otherwise.

The book makes the same point about inventory: "A reward function that only counts this period's holding and shortage cost will produce an agent that orders in spikes" (Sect. 3.4.6). The swings land on the supplier, so the chain as a whole can be worse off while the agent's own numbers look better. The book's remedy is to put a price on order swings, and it is clear that the price is the firm's to set, not the analyst's.

**Run.** The same maze, with five different prices for each muddy square the dog steps on.

In [ ]:
MAZE_3 = ["#########",
          "#S.MMM.T#",
          "#.#####.#",
          "#.......#",
          "#########"]
MUD_PRICES = [0.0, 0.5, 1.0, 2.0, 4.0]    # try changing this: cost per muddy square, on top of the step's cost
results = []
for price in MUD_PRICES:
    m = Maze(MAZE_3, mud_cost=price)
    Q, _ = q_learning(m, trials=800, seed=0)
    path, ok = follow(m, Q)
    results.append((price, ok, len(path) - 1, sum(q in m.mud for q in path), path, m))

print("price of mud | reaches the treat | route length | muddy squares crossed")
for price, ok, length, muddy, _, _ in results:
    print(f"{price:12.1f} | {'yes' if ok else 'no':>17} | {length:12d} | {muddy:d}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, (price, ok, length, muddy, path, m) in zip(axes, (results[0], results[-1])):
    draw(m, paths=[(f"mud price {price}", path)], title=f"Mud price {price}: {'through the mud' if muddy else 'the clean way around'}", ax=ax)
plt.tight_layout(); plt.show()

SHIPPED = MUD_PRICES == [0.0, 0.5, 1.0, 2.0, 4.0]
if SHIPPED:
    assert all(ok for _, ok, *_ in results), "every dog reaches the treat"
    assert [(l, k) for _, _, l, k, _, _ in results] == [(6, 3), (6, 3), (6, 3), (10, 0), (10, 0)], \
        "through the mud up to a price of 1, the clean route from 2"
else:
    print("Checks skipped: you changed the settings.")

**Notice.**
- When mud is free, the dog takes the 6-step shortcut through three muddy squares, and it still does at a price of 0.5 or 1.
- At a price of 2 or more per muddy square, it takes the 10-step clean route around. In this run the switch comes between a price of 1 and 2.
- The dog chose the route, but the *owner* chose the price of mud.

**Discuss.** What does your scorecard leave out: supplier stability, service to the next customer, emissions? Who in your firm should set the price of those side effects?

> **Takeaway.** The reward is a management decision. A learning system optimizes exactly what it is paid for, side effects included.

---
## 5. Before you fund it: the simple rule, and what the learner can see

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

A simple rule beats a learner that cannot see where the treat is: that learner almost never gets there. Give the same learner that one fact and it beats the rule. Before funding RL, measure the rule you already use, and check what the system can see. This maze was built to show that pattern; it is not a test of RL in general.
</details>

**Read.** The book insists that "an AI method must be evaluated against the classical model it displaces" (Chap. 1): the method already in use is the yardstick. So, a contest. In the last maze the treat is in one of two spots, **A** or **B**, a different one each trial. Three contestants:

1. **The simple rule** ("follow your nose"): step to the neighboring square that is fewest blocks from the treat, counting along the grid as if there were no walls, and never turn straight back. It knows where the treat is. It needs no training.
2. **A learner that knows only where it is standing.** It has 2,000 trials to learn.
3. **A learner that also knows which spot the treat is in.** The same method, and the same 2,000 trials.

*This maze was built to show one pattern, the one the book found: a simple rule that is good but not perfect. In a maze where the rule already takes the shortest route, no learner could beat it.*

**Run.** Ten learners of each kind, each tried once with the treat at A and once at B. A trial is stopped at 100 steps.

In [ ]:
MAZE_4 = ["###########",
          "#A.#...#.B#",
          "##.......##",
          "##.#...#.##",
          "#....S....#",
          "###########"]
maze4 = Maze(MAZE_4, max_steps=100)
A, B = maze4.spots
shortest = [shortest_steps(maze4, maze4.start, {g}) for g in (A, B)]
LEARNERS, TRIALS_5 = 10, 2000
outcome = {"Simple rule": [(ok, len(p) - 1, p) for p, ok in (nose_rule(maze4, g, limit=100) for g in (A, B))]}
for label, sees in (("Learner: knows where it is", False), ("Learner: also knows where the treat is", True)):
    runs = []
    for dog in range(LEARNERS):
        Q, _ = q_learning(maze4, trials=TRIALS_5, seed=dog, sees_goal=sees)
        for g in (A, B):
            path, ok = follow(maze4, Q, goal=g, sees_goal=sees, limit=100)
            runs.append((ok, len(path) - 1, path))
    outcome[label] = runs
names = list(outcome)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
reach = [np.mean([ok for ok, _, _ in outcome[k]]) for k in names]
axes[0].barh(names, reach, color=["#757575", "#ef6c00", "#2e7d32"])
axes[0].set_xlim(0, 1.05); axes[0].invert_yaxis(); axes[0].set_xlabel("Share of trials that reach the treat")
axes[0].set_title("The contest")
seeing_B = next(p for (ok, n, p), g in zip(outcome[names[2]], [A, B] * LEARNERS) if g == B)
draw(maze4, paths=[("simple rule", outcome[names[0]][1][2]), ("learner that knows where the treat is", seeing_B)],
     title="Routes when the treat is at B", ax=axes[1], goal=B)
plt.tight_layout(); plt.show()

for k in names:
    hits = [n for ok, n, _ in outcome[k] if ok]
    avg = f"{np.mean(hits):.1f} steps when it does" if hits else "never"
    print(f"{k:40s} reaches the treat in {len(hits)} of {len(outcome[k])} trials; {avg}")
rule_steps = np.mean([n for ok, n, _ in outcome[names[0]]])
seeing_steps = np.mean([n for ok, n, _ in outcome[names[2]]])
print(f"\nShortest possible route: {shortest[0]} steps to A, {shortest[1]} to B.")
print(f"The learner that knows where the treat is takes {100 * (1 - seeing_steps / rule_steps):.0f}% fewer steps than the rule.")

SHIPPED = (LEARNERS, TRIALS_5) == (10, 2000)
if SHIPPED:
    rule_ok = [ok for ok, _, _ in outcome[names[0]]]
    blind_ok = [ok for ok, _, _ in outcome[names[1]]]
    tails = [len(set(p[-6:])) for ok, _, p in outcome[names[1]] if not ok]
    assert all(rule_ok) and rule_steps > np.mean(shortest), "the rule reaches the treat, but detours"
    assert sum(blind_ok) <= 2, "the learner that knows only where it is almost never reaches the treat"
    assert all(t in (1, 2) for t in tails), "its failed trials end standing still or shuttling between two squares"
    assert all(ok for ok, _, _ in outcome[names[2]]) and seeing_steps == np.mean(shortest) < rule_steps, \
        "the informed learner takes the shortest route every time, and beats the rule"
else:
    print("Checks skipped: you changed the settings.")

**Notice.**
- The simple rule reaches the treat both times with no training, though not by the shortest route: it detours around a wall.
- The learner that knows only where it is standing almost never reaches the treat. It learned its habit while the treat kept moving, so its habit is a muddle: in its failed trials it ends up shuttling back and forth between two squares, or pushing against a wall, until it is stopped.
- Give the same method the one fact the decision needs (which spot the treat is in), and it takes the shortest route every time, with fewer steps than the rule.

The book found the same pattern on a delivery grid. (A *tabular* agent keeps a table of scores, like our dog; an *episode* is a trial.) "A tabular Q-learning agent, trained for 3,000 episodes on the grid delivery environment, is beaten by a rule that fits on one line — move toward the nearest undelivered customer — and beaten by more than a factor of ten." Then "the same algorithm, given a state that carries what the decision needs … beats the rule by 7.2%" (Chap. 5, *Model Performance Assessment*).

**Discuss.** What is "where the treat is" in your business: the promotion calendar, a supplier's real lead time, today's traffic? Does your data give the system that information when it decides?

> **Takeaway.** Measure the simple rule first: it is often better than it looks. And before buying more training, check what the system can *see*: practice cannot supply a fact the system never sees.

---
## 6. From the maze back to the supply chain

<details><summary><b>The two-minute version</b> (also the script for the audio summary)</summary>

Six questions before you fund RL: Is it a sequence? Where will it practice? What is it paid for? What does it replace? What can it see? How much exploring can you afford?
</details>

| In the maze | In inventory (Sect. 3.4) | In Chap. 5's delivery grid |
|---|---|---|
| The dog | The ordering system | The delivery vehicle |
| Where it stands | Stock on hand, orders on the way (each order separately), recent demand, the forecast, the lead-time estimate, the calendar | Its square and the deliveries remaining; in the better version, the direction of the nearest delivery and the congestion nearby |
| Which way to go | How much to order | Which way to drive next |
| Treat, minus a cost per step | Minus holding, shortage and ordering cost | +10 per delivery, −1 per step |
| The price of mud | The price of order swings that hurt the supplier (Sect. 3.4.6) | The −2 penalty for a congested square |
| Practicing in the maze | Training in a simulator (Sect. 3.4.4) | Training on the simulated grid |

Chapter 8 goes one step further and sets out how responses to disruptions could be learned inside a *digital twin*, a simulation kept in step with the real operation ("formulated, not trained", as Appendix A puts it).

### When is RL worth it? A manager's checklist

1. **Is it a sequence?** Does today's decision change tomorrow's situation? If each decision stands alone, simpler methods will do.
2. **Where will it practice?** Is there a simulator you trust enough to make its mistakes in?
3. **What is it paid for?** Does the reward price the side effects you care about, and who set those prices?
4. **What does it replace?** Has the current rule been measured on the same test?
5. **What can it see?** Does it get the information the decision actually needs?
6. **How much exploring can you afford,** and where?

### Questions to ask your analytics team
- What is the baseline policy, and what did it score on the same simulated demand?
- How many trials did training take, and in what simulator? How was the simulator checked against reality?
- What exactly is the reward? What does it leave out?
- What information does the agent see when it decides? What would it need that it does not have?
- How will we know, after launch, that it still beats the baseline?

---
## 7. Check yourself

<details><summary><b>1.</b> In one sentence, how does reinforcement learning differ from learning from past examples?</summary>

Learning from examples copies known right answers; reinforcement learning has no right answers to copy, only a score after acting, and each decision changes the next situation (Parts 1–2).
</details>

<details><summary><b>2.</b> Why do the early trials matter to a manager?</summary>

They are the bad ones. Paying for them on the live business means stockouts or overstock, which is why the learning is done in a simulator (Part 2).
</details>

<details><summary><b>3.</b> A system that never explores settled for the snack. What is the business version, and what does exploring cost?</summary>

Keeping the first option that works (a supplier, a policy). Exploring costs trials that earn little; the question is how much to spend, and where (Part 3).
</details>

<details><summary><b>4.</b> Who should set the price of mud, and why?</summary>

The business, because it is a trade-off between local cost and a side effect such as supplier stability; the system cannot know what that is worth to the firm (Part 4).
</details>

<details><summary><b>5.</b> Why would more training not help the learner that knew only where it stood?</summary>

It lacked the one fact the decision needed: which spot the treat was in. Practice cannot supply information the system never sees (Part 5).
</details>

### If your instructor sets it: a one-page decision memo

Priya's vendor proposes an RL system for weekly replenishment of 400 fresh-food items *(a constructed example)*. Write Priya a one-page memo recommending **fund a simulated pilot**, **fund a live pilot**, or **decline for now**. Use the checklist in Part 6, say which evidence you would ask the vendor for, and say what result would change your mind.

---
## Where to go next

- **The same ideas, with the formulas:** the students' intro *Reinforcement Learning: A First Look* (slot machines and a robot in a grid).
- **In the book:** Appendix A.4 introduces the idea in a business setting. Sects. 3.4.2–3.4.6 cover the inventory agent's state and reward, the simulator, the baseline and the price of order swings; Chap. 5's *Model Performance Assessment* reports the delivery-grid contest; Sect. 8.4 sets out learning inside a digital twin.
- **Readings from the book's reference lists:** Cachon, G., & Terwiesch, C. (2018). *Matching Supply with Demand: An Introduction to Operations Management* (4th ed.). McGraw-Hill (listed in Chaps. 1, 3 and 9), for the inventory basics behind Part 6. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press (listed in Chaps. 3 and 8), the field's textbook, for those who want the method in full.

---
## How to cite

Reinforcement Learning: A First Look (Version 1.0.0) [Computer software]. Zenodo. https://doi.org/10.5281/zenodo.23968091

---
## License (code only)

The code cells of this notebook are under the MIT License, the same license as the book's code. The explanatory text is not.

```
MIT License

Copyright (c) 2026 Charles C. Luo

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```